# FinGPT Model Server

This notebook loads FinGPT (Llama2-7B + LoRA adapter, 4-bit quantized) and
exposes it as an HTTP API via ngrok so your local backend can call it.

**Prerequisites:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add secrets in the left sidebar (key icon):
   - `HF_TOKEN` — your Hugging Face token
   - `NGROK_AUTH_TOKEN` — your ngrok auth token

Run all cells, then copy the ngrok URL into your local `config/.env`.

In [ ]:
# Cell 1: Install dependencies (updated for Colab 2026)
!pip install -q --upgrade bitsandbytes
!pip install -q transformers accelerate peft torch sentencepiece protobuf flask pyngrok triton

In [ ]:
# Cell 2: Check GPU
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3: Load secrets
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

print('Secrets loaded OK' if HF_TOKEN and NGROK_AUTH_TOKEN else 'ERROR: Add secrets in sidebar!')

In [ ]:
# Cell 4: Load FinGPT model (4-bit quantized Llama2-7B + LoRA)
# This takes ~2-4 minutes on first run (downloading ~3.5GB)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 4-bit quantization config — fits in T4's 16GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

BASE_MODEL = 'meta-llama/Llama-2-7b-chat-hf'
PEFT_MODEL = 'FinGPT/fingpt-mt_llama2-7b_lora'

print(f'Loading base model: {BASE_MODEL} (4-bit)...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
    trust_remote_code=True,
)

print(f'Loading FinGPT LoRA adapter: {PEFT_MODEL}...')
model = PeftModel.from_pretrained(model, PEFT_MODEL)
model.eval()

# Check VRAM usage
mem = torch.cuda.memory_allocated() / 1e9
print(f'\nModel loaded! VRAM used: {mem:.1f} GB')
print('Ready for inference.')

In [ ]:
# Cell 5: Define inference functions

def run_inference(prompt: str, max_new_tokens: int = 256) -> str:
    """Run a single prompt through the model."""
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.1,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = full_text[len(tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)):].strip()
    return response


SENTIMENT_PROMPT = """Instruction: What is the sentiment of this news? Please choose an answer from {{negative/neutral/positive}}
Input: {text}
Answer: """

HEADLINE_PROMPT = """Instruction: Does the news headline talk about price going up? Please choose an answer from {{Yes/No}}.
Input: {text}
Answer: """

INSIGHT_PROMPT = """Instruction: You are a financial analyst. Based on the following portfolio data and market conditions, provide a brief risk assessment and investment insight in 3-4 sentences.
Input: {text}
Answer: """


def analyze(text: str, task: str = 'sentiment') -> dict:
    if task == 'sentiment':
        prompt = SENTIMENT_PROMPT.format(text=text)
        raw = run_inference(prompt, max_new_tokens=10)
        sentiment = 'neutral'
        for s in ['positive', 'negative', 'neutral']:
            if s in raw.lower():
                sentiment = s
                break
        return {'sentiment': sentiment, 'raw': raw}

    elif task == 'headline':
        prompt = HEADLINE_PROMPT.format(text=text)
        raw = run_inference(prompt, max_new_tokens=10)
        direction = 'No'
        if 'yes' in raw.lower():
            direction = 'Yes'
        return {'direction': direction, 'raw': raw}

    elif task == 'insight':
        prompt = INSIGHT_PROMPT.format(text=text)
        raw = run_inference(prompt, max_new_tokens=256)
        return {'insight': raw}

    else:
        return {'error': f'Unknown task: {task}'}


# Quick test
test = analyze('Apple beats earnings expectations with strong iPhone sales', 'sentiment')
print(f'Test result: {test}')

In [ ]:
# Cell 6: Start Flask server + ngrok tunnel

from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

flask_app = Flask(__name__)

@flask_app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'model': 'FinGPT-MT-Llama2-7B-LoRA', 'quantization': '4bit'})

@flask_app.route('/analyze', methods=['POST'])
def analyze_endpoint():
    data = request.json
    text = data.get('text', '')
    task = data.get('task', 'sentiment')
    if not text:
        return jsonify({'error': 'No text provided'}), 400
    result = analyze(text, task)
    return jsonify(result)

@flask_app.route('/batch_analyze', methods=['POST'])
def batch_analyze():
    data = request.json
    headlines = data.get('headlines', [])
    task = data.get('task', 'sentiment')
    results = []
    for h in headlines:
        try:
            results.append(analyze(h, task))
        except Exception as e:
            results.append({'error': str(e), 'headline': h})
    return jsonify({'results': results})

# Setup ngrok with your free static domain
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# PASTE YOUR DOMAIN BELOW (from https://dashboard.ngrok.com/domains)
YOUR_DOMAIN = "offertorial-rudy-nonpertinently.ngrok-free.dev"  # <-- CHANGE THIS

public_url = ngrok.connect(5000, domain=YOUR_DOMAIN)

print('='*60)
print(f'FinGPT Model Server is LIVE!')
print(f'')
print(f'  Public URL: https://{YOUR_DOMAIN}')
print(f'')
print(f'  Copy this URL into your config/.env as COLAB_MODEL_URL')
print('='*60)

threading.Thread(target=lambda: flask_app.run(port=5000), daemon=True).start()
print('\nServer running in background. Keep this notebook open.')

In [ ]:
# Cell 7 (optional): Test the server locally before using from your ThinkPad
import requests

# Test health
r = requests.get('http://localhost:5000/health')
print('Health:', r.json())

# Test sentiment
r = requests.post('http://localhost:5000/analyze', json={
    'text': 'Federal Reserve raises interest rates by 25 basis points',
    'task': 'sentiment'
})
print('Sentiment:', r.json())

# Test headline classification
r = requests.post('http://localhost:5000/analyze', json={
    'text': 'Tesla stock surges 15% on record deliveries',
    'task': 'headline'
})
print('Headline:', r.json())

## Done!

The model server is running. Now on your ThinkPad:

1. Copy the ngrok URL above
2. Paste it into `config/.env` as `COLAB_MODEL_URL=https://xxxx.ngrok-free.app`
3. Run `python backend/app.py`
4. Open http://localhost:8000

**Remember to disconnect this runtime when done to save compute units!**